# Automatic engineering discipline orchestration

This notebook demonstrates the post-#2460 execution layer: explicit policy configuration, fail-closed coverage, dependency fingerprints, revision impact and coordinated multi-area packaging. The values are illustrative preliminary inputs and all results remain review-required.

In [ ]:
from pathlib import Path
import jpype
import jpype.imports

ROOT = Path.cwd().resolve()
while ROOT.name != 'neqsim' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not jpype.isJVMStarted():
    jpype.startJVM(classpath=[str(ROOT / 'target' / 'classes')])


Build and run a normal NeqSim `ProcessSystem`, then create the governed project and controlled cases as shown in `process_to_engineering_simulator.ipynb`. The policy below contains explicit values instead of library screening defaults.

In [ ]:
Policy = jpype.JClass('neqsim.process.engineering.production.EngineeringAutoConfigurationPolicy')
Simulator = jpype.JClass('neqsim.process.engineering.ProcessToEngineeringSimulator')

policy = Policy('offshore-gas', 'A').addInletCompressionExportSlice(
    'INLET-SEP', 'EXPORT-COMP', 'EXPORT-LINE', '', 'PIT-100',
    800.0, 0.107, 120.0, 25.0, 5.0, 0.10,
    500.0, 1000.0, 2000.0, 3000.0, 5000.0, 7500.0, 10000.0)
# result = Simulator.run(project, policy, 2)
# orchestration = project.getProductionReadinessBasis().getAutoConfigurationResult()
# assert orchestration.isExecutionReady(), list(orchestration.getExecutionBlockers())
# print(orchestration.getConfigurationFingerprint())
# print(dict(orchestration.getModuleDependencies()))


## Revision and multi-area workflow

Store the prior `EngineeringAutoConfigurator.Result` with the revision. Call `current.compareWith(previous)` before reuse; regenerate every listed artifact when the fingerprint changes. For a `ProcessModel`, create one `ProcessModelEngineeringSimulator.AreaConfiguration` per area, attach controlled cases, call `run`, and then `compile`. The root manifest records area package paths and shared-stream invalidation links. Missing area policies and incomplete equipment coverage are blockers, not warnings.

## Engineering boundary

Successful execution does not establish credible HAZOP scenarios, SIL/voting, final set points, valve failure positions, metallurgy, utility simultaneity, flare/blowdown concurrency, vendor guarantees or final approval. Supply those through controlled project evidence and qualification workflows.